In [1]:
"""
Preprocessing pipeline — NMDC datasets

This module is responsible for:
1. Walking the raw NMDC study/category/biosample directory tree
2. Extracting the biosample identifier (nmdc_bsm-...) from each file path
3. Parsing table (CSV/TSV), FASTA and GFF files into DataFrames
4. Writing one parquet file per biosample/category into parquet_output_samples

Only the 3 categories actually consumed by the downstream pipeline
(EC, KO, GC-MS) are processed here. Any other category present in the
raw NMDC download is skipped.

IMPORTANT:
- This notebook only needs to be re-run if you want to regenerate
  parquet_output_samples from a fresh raw NMDC download
- The repository already ships with parquet_output_samples pre-built
  (data/parquet_output_samples), so this step is NOT required to run
  the rest of the pipeline
- DATASET_ROOT (raw input) is not versioned in the repository — point
  it at wherever you downloaded the raw NMDC study data for STUDY_ID

Author: Adriany Adila
"""

import os
import subprocess
import pandas as pd
from collections import defaultdict
from pathlib import Path

# REPOSITORY SETUP

REPO_URL = "https://github.com/adrianyadila/ML-BIO-TECH.git"
REPO_DIR = "/content/ML-BIO-TECH"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

PROJECT_ROOT = REPO_DIR

# CONFIGURATION

# Study to process — required, one study per run
STUDY_ID = "sty-11-aygzgv51"

# Raw NMDC download (not part of the repository)
DATASET_ROOT = "/content/drive/MyDrive/ML BIO TECH/dataset"

# Output — lives inside the cloned repository
PARQUET_ROOT = os.path.join(PROJECT_ROOT, "data", "parquet_output_samples")

# Only the categories actually used by the downstream pipeline
CATEGORY_EXTENSIONS = {
    "Annotation_KEGG_Orthology": [".tsv", ".csv"],
    "Annotation_Enzyme_Commission": [".tsv", ".csv"],
    "GC_MS_Metabolomics_Results": [".csv"],
}

# UTILITIES

def extract_biosample_from_path(path: str) -> str:
    """Extracts the biosample identifier (nmdc_bsm-...) from a file path."""
    for part in path.split(os.sep):
        if part.startswith("nmdc_bsm-"):
            return part
    raise ValueError(f"Biosample not found in path: {path}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


# FILE PROCESSORS

def process_table_file(file_path: str) -> pd.DataFrame:
    if file_path.endswith(".csv"):
        return pd.read_csv(file_path)
    elif file_path.endswith(".tsv"):
        return pd.read_csv(file_path, sep="\t")
    else:
        raise ValueError(f"Unsupported table format: {file_path}")


def process_fasta_file(file_path: str) -> pd.DataFrame:
    records = []
    with open(file_path) as f:
        current = None
        seq = []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current:
                    records.append({
                        "id": current,
                        "sequence": "".join(seq)
                    })
                current = line[1:]
                seq = []
            else:
                seq.append(line)
        if current:
            records.append({
                "id": current,
                "sequence": "".join(seq)
            })
    return pd.DataFrame(records)


def process_gff_file(file_path: str) -> pd.DataFrame:
    records = []
    with open(file_path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 9:
                records.append({
                    "seqid": parts[0],
                    "source": parts[1],
                    "type": parts[2],
                    "start": parts[3],
                    "end": parts[4],
                    "score": parts[5],
                    "strand": parts[6],
                    "phase": parts[7],
                    "attributes": parts[8],
                })
    return pd.DataFrame(records)


# MAIN

if __name__ == "__main__":

    dataset_dir = DATASET_ROOT
    parquet_root = PARQUET_ROOT

    counters = defaultdict(int)

    # LOOP 1 — STUDY (limited to the selected STUDY_ID)
    study_dirs = [
        d for d in os.listdir(dataset_dir)
        if d == f"nmdc_{STUDY_ID}"
        and os.path.isdir(os.path.join(dataset_dir, d))
    ]

    if not study_dirs:
        raise RuntimeError(f"Study nmdc_{STUDY_ID} not found in dataset.")

    for study_folder in study_dirs:

        study_path = os.path.join(dataset_dir, study_folder)
        study_name = STUDY_ID  # fixed, no inference

        print(f"\n=== Processing study: {study_name} ===")

        # LOOP 2 — CATEGORY (within the study)
        for category_folder in os.listdir(study_path):

            category_path = os.path.join(study_path, category_folder)

            if not os.path.isdir(category_path):
                continue

            # Expected: nmdc_<study>_<CATEGORY_NAME>
            if not category_folder.startswith(f"nmdc_{STUDY_ID}_"):
                continue

            category_name = category_folder.replace(
                f"nmdc_{STUDY_ID}_", ""
            )

            if category_name not in CATEGORY_EXTENSIONS:
                print(f"Category ignored: {category_name}")
                continue

            print(f"\nProcessing category: {category_name}")

            # LOOP 3 — FILES (recursive)
            for root, _, files in os.walk(category_path):

                for file in files:

                    if not any(
                        file.endswith(ext)
                        for ext in CATEGORY_EXTENSIONS[category_name]
                    ):
                        continue

                    file_path = os.path.join(root, file)

                    try:
                        biosample = extract_biosample_from_path(file_path)
                    except ValueError:
                        # Files without a biosample are skipped
                        continue

                    try:
                        if file.endswith((".csv", ".tsv")):
                            df = process_table_file(file_path)
                        elif file.endswith((".fna", ".faa", ".fasta")):
                            df = process_fasta_file(file_path)
                        elif file.endswith(".gff"):
                            df = process_gff_file(file_path)
                        else:
                            continue
                    except Exception as e:
                        print(f"Failed processing {file_path}: {e}")
                        continue

                    # OUTPUT — always organized by biosample
                    out_dir = os.path.join(
                        parquet_root,
                        category_name,
                        study_name,
                        biosample
                    )

                    ensure_dir(out_dir)

                    out_file = os.path.join(
                        out_dir,
                        f"part{counters[(category_name, biosample)]:05d}.parquet"
                    )

                    # No overwrite
                    if os.path.exists(out_file):
                        continue

                    df.to_parquet(out_file, index=False)
                    counters[(category_name, biosample)] += 1

    print("\nPreprocessing finished.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ML BIO TECH/dataset'